# Notebook 03 Phase 1B Model Comparison Readiness

This notebook is the Phase 1B model comparison entrypoint for LSTM, TCN, and standard DLinear baselines under no-trade-band binary labels.

Default mode is inspection-only: no training, no checkpoint writing, and no result artifact writing. Full execution requires explicit guard changes in a later approved step.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path
import copy
import json
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

FULL_RUN = False
RUN_TRAINING = False
WRITE_ARTIFACTS = False
ALLOW_OVERWRITE = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_ROOT = Path("/content/drive/MyDrive/stockdata")
RAW_DATA_DIR = DATA_ROOT / "Dow_30_1min"
OUTPUT_DIR = DATA_ROOT / "phase1b_notebook03_model_comparison"

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from ml_utils.config import DataConfig
from ml_utils.dataset import WindowedClassificationDataset
from ml_utils.dataset import fit_scaler_on_train
from ml_utils.dataset import make_no_trade_band_labels
from ml_utils.dataset import make_time_splits
from ml_utils.dataset import transform_split
from ml_utils.dataset import trim_labels_at_split_boundary
from ml_utils.metrics import always_predict_baseline_metrics
from ml_utils.metrics import compute_classification_metrics
from ml_utils.metrics import dummy_baseline_metrics
from ml_utils.models.lstm_classifier import LSTMClassifier
from ml_utils.models.tcn_classifier import TCNClassifier
from ml_utils.models.dlinear_classifier import DLinearClassifier
from ml_utils.seed import seed_everything
from ml_utils.trainer import evaluate
from ml_utils.trainer import train_one_epoch

In [ ]:
ALL_TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
ALL_SEEDS = [42, 43, 44]

CANDIDATE_GRID = [
    {
        "candidate_id": "A",
        "candidate_name": "main",
        "window_size": 12,
        "label_horizon_k": 12,
        "threshold_bps": 5,
    },
    {
        "candidate_id": "B",
        "candidate_name": "secondary_window24",
        "window_size": 24,
        "label_horizon_k": 12,
        "threshold_bps": 5,
    },
    {
        "candidate_id": "C",
        "candidate_name": "longer_horizon24",
        "window_size": 12,
        "label_horizon_k": 24,
        "threshold_bps": 5,
    },
    {
        "candidate_id": "D",
        "candidate_name": "window24_horizon24",
        "window_size": 24,
        "label_horizon_k": 24,
        "threshold_bps": 5,
    },
]

SELECTED_CANDIDATES = ["A"]
SELECTED_MODELS = ["lstm"]
SELECTED_TICKERS = ["CSCO"]
SELECTED_SEEDS = [42]
MAX_RAW_ROWS_PER_TICKER = 20_000
MAX_EPOCHS = 1

TICKERS = ALL_TICKERS
SEEDS = ALL_SEEDS
DEFAULT_CANDIDATE = CANDIDATE_GRID[0]
WINDOW_SIZE = DEFAULT_CANDIDATE["window_size"]
LABEL_HORIZON_K = DEFAULT_CANDIDATE["label_horizon_k"]
THRESHOLD_BPS = DEFAULT_CANDIDATE["threshold_bps"]

NUM_CLASSES = 2
INPUT_SIZE = 12

BATCH_SIZE = 512
LEARNING_RATE = 1e-4
EARLY_STOP_PATIENCE = 5
SUSPICIOUS_MACRO_F1_THRESHOLD = 0.90

TIMESTAMP_COL = "timestamp"
TICKER_COL = "ticker"
LABEL_COL = "label"
PRICE_COL = "close"
SCALER_TYPE = "standard"

RAW_COLUMNS = ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]

FEATURE_COLS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "macd",
    "macd_signal",
    "macd_hist",
    "rsi_14",
    "bb_pctb",
    "rolling_std_20",
    "obv_roc",
]

MARKET_OPEN = "09:30"
MARKET_CLOSE = "16:00"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
candidate_grid_df = pd.DataFrame(CANDIDATE_GRID)
candidate_grid_df

## Historical LSTM Context

P1B.10 Candidate A single-seed result:

- `model_macro_f1_mean` ≈ 0.5203
- `dummy_stratified_macro_f1_mean` ≈ 0.4990
- delta ≈ +0.0213

P1B.11b strict multi-seed result:

- seed 42 delta ≈ +0.0213
- seed 43 delta ≈ +0.0038
- seed 44 delta ≈ -0.0089
- overall mean delta ≈ +0.0054
- std ≈ 0.0152

Interpretation: LSTM is weak, seed-sensitive, and not robustly better than dummy. These values are historical context, not Notebook 03 output.

In [ ]:
MODEL_REGISTRY = [
    {
        "model_name": "lstm",
        "import_path": "ml_utils.models.lstm_classifier.LSTMClassifier",
        "constructor": LSTMClassifier,
        "config": {
            "hidden_size": 64,
            "num_layers": 2,
            "dropout": 0.2,
        },
    },
    {
        "model_name": "tcn",
        "import_path": "ml_utils.models.tcn_classifier.TCNClassifier",
        "constructor": TCNClassifier,
        "config": {
            "num_channels": [32, 32],
            "kernel_size": 3,
            "dropout": 0.1,
            "causal": True,
        },
    },
    {
        "model_name": "dlinear",
        "import_path": "ml_utils.models.dlinear_classifier.DLinearClassifier",
        "constructor": DLinearClassifier,
        "config": {
            "moving_avg_kernel": 5,
            "individual": False,
        },
    },
]


def model_config_for(model_name: str, candidate_config: dict) -> dict:
    registry_by_name = {item["model_name"]: item for item in MODEL_REGISTRY}
    if model_name not in registry_by_name:
        raise ValueError(f"Unknown model_name {model_name!r}; expected one of {sorted(registry_by_name)}")

    config = dict(registry_by_name[model_name]["config"])
    config["input_size"] = INPUT_SIZE
    config["num_classes"] = NUM_CLASSES
    if model_name == "dlinear":
        config["seq_len"] = int(candidate_config["window_size"])
    return config

In [ ]:
model_registry_df = pd.DataFrame(
    [
        {
            "model_name": item["model_name"],
            "import_path": item["import_path"],
            "config": item["config"],
        }
        for item in MODEL_REGISTRY
    ]
)
model_registry_df


In [ ]:
NOTEBOOK03_RESULT_COLUMNS = [
    "candidate_id",
    "candidate_name",
    "model_name",
    "ticker",
    "seed",
    "window_size",
    "label_horizon_k",
    "threshold_bps",
    "split",
    "n_train_windows",
    "n_val_windows",
    "n_test_windows",
    "label_retained_pct",
    "model_macro_f1",
    "model_balanced_accuracy",
    "dummy_stratified_macro_f1_mean",
    "dummy_stratified_macro_f1_std",
    "dummy_prior_macro_f1",
    "always_up_macro_f1",
    "always_down_macro_f1",
    "delta_macro_f1_vs_dummy",
    "confusion_matrix",
]

In [ ]:
results_schema_df = pd.DataFrame(columns=NOTEBOOK03_RESULT_COLUMNS)
results_schema_df


In [ ]:
ARTIFACT_NAMES = {
    "per_ticker_results": "notebook03_per_ticker_results.csv",
    "summary_by_model": "notebook03_summary_by_model.csv",
    "summary_by_seed": "notebook03_summary_by_seed.csv",
    "run_manifest": "notebook03_run_manifest.json",
}

RUN_MANIFEST_TEMPLATE = {
    "notebook": "03_model_comparison.ipynb",
    "phase": "P1B.20b",
    "timestamp": None,
    "git_commit_hash": None,
    "guards": {
        "full_run": FULL_RUN,
        "run_training": RUN_TRAINING,
        "write_artifacts": WRITE_ARTIFACTS,
        "allow_overwrite": ALLOW_OVERWRITE,
    },
    "scope": {
        "selected_candidates": SELECTED_CANDIDATES,
        "selected_models": SELECTED_MODELS,
        "selected_tickers": SELECTED_TICKERS,
        "selected_seeds": SELECTED_SEEDS,
        "max_raw_rows_per_ticker": MAX_RAW_ROWS_PER_TICKER,
        "max_epochs": MAX_EPOCHS,
    },
    "artifact_paths": {},
}

In [ ]:
artifact_manifest_df = pd.DataFrame(
    [{"artifact_key": key, "filename": value} for key, value in ARTIFACT_NAMES.items()]
)
artifact_manifest_df


In [ ]:
def raw_path_for_ticker(ticker: str) -> Path:
    return RAW_DATA_DIR / f"{ticker}.txt"


def load_one_minute_data(ticker: str, max_rows: int | None = None) -> pd.DataFrame:
    input_path = raw_path_for_ticker(ticker)
    if not input_path.exists():
        raise FileNotFoundError(f"Missing raw 1-minute txt file: {input_path}")

    nrows = max_rows if max_rows is None or max_rows > 0 else 0
    frame = pd.read_csv(input_path, header=None, names=RAW_COLUMNS, nrows=nrows)
    frame = frame[frame["Date"].astype(str).str.lower() != "date"].reset_index(drop=True)
    frame[TIMESTAMP_COL] = pd.to_datetime(
        frame["Date"].astype(str) + " " + frame["Time"].astype(str),
        format="%m/%d/%Y %H:%M",
    )
    frame = frame.drop(columns=["Date", "Time"])
    frame = frame.rename(
        columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    numeric_cols = ["open", "high", "low", "close", "volume"]
    frame[numeric_cols] = frame[numeric_cols].apply(pd.to_numeric, errors="raise")
    return frame[[TIMESTAMP_COL, *numeric_cols]].reset_index(drop=True)


In [ ]:
def filter_regular_hours(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy(deep=True)
    if TIMESTAMP_COL not in result.columns:
        raise ValueError(f"Missing required timestamp column: {TIMESTAMP_COL}")
    result[TIMESTAMP_COL] = pd.to_datetime(result[TIMESTAMP_COL])
    market_open = pd.to_datetime(MARKET_OPEN).time()
    market_close = pd.to_datetime(MARKET_CLOSE).time()
    times = result[TIMESTAMP_COL].dt.time
    regular_hours = result[(times >= market_open) & (times <= market_close)]
    return regular_hours.reset_index(drop=True)


def resample_to_five_minutes(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = [TIMESTAMP_COL, "open", "high", "low", "close", "volume"]
    missing_cols = [column for column in required_cols if column not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns for 5-minute resampling: {missing_cols}")

    resampled = (
        df.copy(deep=True)
        .set_index(TIMESTAMP_COL)
        .resample("5min")
        .agg(
            {
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last",
                "volume": "sum",
            }
        )
        .dropna(subset=["open", "high", "low", "close", "volume"])
        .reset_index()
    )
    return filter_regular_hours(resampled)


In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy(deep=True)

    ema12 = result["close"].ewm(span=12, adjust=False).mean()
    ema26 = result["close"].ewm(span=26, adjust=False).mean()
    result["macd"] = ema12 - ema26
    result["macd_signal"] = result["macd"].ewm(span=9, adjust=False).mean()
    result["macd_hist"] = result["macd"] - result["macd_signal"]

    close_delta = result["close"].diff()
    gain = close_delta.clip(lower=0)
    loss = (-close_delta).clip(lower=0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs = avg_gain / avg_loss
    result["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))

    rolling_mean_20 = result["close"].rolling(20).mean()
    rolling_std_price_20 = result["close"].rolling(20).std()
    upper_band = rolling_mean_20 + 2.0 * rolling_std_price_20
    lower_band = rolling_mean_20 - 2.0 * rolling_std_price_20
    result["bb_pctb"] = (result["close"] - lower_band) / (upper_band - lower_band)
    result["rolling_std_20"] = result["close"].pct_change().rolling(20).std()

    obv_direction = np.sign(result["close"].diff())
    obv_direction.iloc[0] = 0
    obv = (obv_direction * result["volume"]).cumsum()
    result["obv_roc"] = obv.pct_change(5)

    result[FEATURE_COLS] = result[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    return result.dropna(subset=FEATURE_COLS).reset_index(drop=True)


In [ ]:
def prepare_ticker_frame(ticker: str, max_raw_rows: int | None = None) -> pd.DataFrame:
    one_minute = load_one_minute_data(ticker, max_rows=max_raw_rows)
    regular_hours = filter_regular_hours(one_minute)
    five_minute = resample_to_five_minutes(regular_hours)
    feature_frame = add_technical_indicators(five_minute)
    if feature_frame.empty:
        raise ValueError(
            f"Ticker {ticker} has no usable rows after 5-minute resampling and feature cleanup"
        )
    feature_frame[TICKER_COL] = ticker
    return feature_frame


def apply_no_trade_band_labels(
    df: pd.DataFrame,
    candidate_config: dict,
) -> tuple[pd.DataFrame, dict]:
    return make_no_trade_band_labels(
        df,
        price_col=PRICE_COL,
        k=int(candidate_config["label_horizon_k"]),
        threshold_bps=float(candidate_config["threshold_bps"]),
        timestamp_col=TIMESTAMP_COL,
    )

In [ ]:
def _make_window_dataset(
    frame: pd.DataFrame,
    candidate_config: dict,
) -> WindowedClassificationDataset:
    return WindowedClassificationDataset(
        df=frame,
        feature_cols=FEATURE_COLS,
        label_col=LABEL_COL,
        ticker_col=TICKER_COL,
        timestamp_col=TIMESTAMP_COL,
        window_size=int(candidate_config["window_size"]),
        label_horizon_k=int(candidate_config["label_horizon_k"]),
        stride=1,
    )


def prepare_model_data(
    tickers: list[str],
    candidate_config: dict,
    max_raw_rows_per_ticker: int | None = None,
):
    if not tickers:
        raise ValueError("tickers must be non-empty")

    data_config = DataConfig(
        tickers=tickers,
        data_dir=str(RAW_DATA_DIR),
        timestamp_col=TIMESTAMP_COL,
        price_col=PRICE_COL,
        label_mode="no_trade_band",
        threshold_bps=float(candidate_config["threshold_bps"]),
        feature_cols=FEATURE_COLS,
        train_ratio=0.70,
        val_ratio=0.15,
        timezone_policy="naive",
    )

    labeled_frames = {}
    label_diagnostics = {}
    split_frames = {"train": {}, "val": {}, "test": {}}

    for ticker in tickers:
        feature_frame = prepare_ticker_frame(ticker, max_raw_rows=max_raw_rows_per_ticker)
        labeled_frame, diagnostics = apply_no_trade_band_labels(feature_frame, candidate_config)
        labeled_frames[ticker] = labeled_frame
        label_diagnostics[ticker] = diagnostics

        train_frame, val_frame, test_frame = make_time_splits(
            labeled_frame,
            train_ratio=data_config.train_ratio,
            val_ratio=data_config.val_ratio,
            timestamp_col=TIMESTAMP_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["train"][ticker] = trim_labels_at_split_boundary(
            train_frame,
            label_horizon_k=int(candidate_config["label_horizon_k"]),
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["val"][ticker] = trim_labels_at_split_boundary(
            val_frame,
            label_horizon_k=int(candidate_config["label_horizon_k"]),
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["test"][ticker] = trim_labels_at_split_boundary(
            test_frame,
            label_horizon_k=int(candidate_config["label_horizon_k"]),
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )

    pooled_train_frame = pd.concat(
        [split_frames["train"][ticker] for ticker in tickers],
        ignore_index=True,
    )
    scaler = fit_scaler_on_train(pooled_train_frame, FEATURE_COLS, scaler_type=SCALER_TYPE)

    transformed = {"train": {}, "val": {}, "test": {}}
    datasets = {"train": {}, "val": {}, "test": {}}
    labels = {"train": {}, "val": {}, "test": {}}
    for split_name in ["train", "val", "test"]:
        for ticker in tickers:
            transformed[split_name][ticker] = transform_split(
                split_frames[split_name][ticker],
                scaler,
                FEATURE_COLS,
            )
            datasets[split_name][ticker] = _make_window_dataset(
                transformed[split_name][ticker],
                candidate_config,
            )
            labels[split_name][ticker] = collect_dataset_labels(datasets[split_name][ticker])

    return {
        "data_config": data_config,
        "candidate_config": candidate_config,
        "labeled_frames": labeled_frames,
        "label_diagnostics": label_diagnostics,
        "split_frames": split_frames,
        "transformed": transformed,
        "datasets": datasets,
        "labels": labels,
        "scaler": scaler,
    }

In [ ]:
from torch.utils.data import DataLoader


def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )


def collect_dataset_labels(dataset) -> np.ndarray:
    labels = [int(dataset[index][1].item()) for index in range(len(dataset))]
    return np.asarray(labels, dtype=np.int64)


def class_distribution(labels: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.int64)
    count = int(labels.size)
    up = int((labels == 1).sum())
    down = int((labels == 0).sum())
    return {
        "n": count,
        "up": up,
        "down": down,
        "up_pct": float(up / count) if count else np.nan,
        "down_pct": float(down / count) if count else np.nan,
    }


In [ ]:
def build_model(model_name: str, candidate_config: dict):
    registry_by_name = {item["model_name"]: item for item in MODEL_REGISTRY}
    if model_name not in registry_by_name:
        raise ValueError(f"Unknown model_name {model_name!r}; expected one of {sorted(registry_by_name)}")
    registry_item = registry_by_name[model_name]
    return registry_item["constructor"](**model_config_for(model_name, candidate_config))

In [ ]:
def summarize_dummy_baselines(y_train: np.ndarray, y_target: np.ndarray, seed: int) -> dict:
    stratified_scores = []
    for offset in range(10):
        metrics = dummy_baseline_metrics(
            y_train=y_train,
            y_test=y_target,
            strategy="stratified",
            random_state=seed + offset,
        )
        stratified_scores.append(metrics["macro_f1"])

    prior_metrics = dummy_baseline_metrics(
        y_train=y_train,
        y_test=y_target,
        strategy="prior",
        random_state=seed,
    )
    always_up_metrics = always_predict_baseline_metrics(y_target, constant_label=1)
    always_down_metrics = always_predict_baseline_metrics(y_target, constant_label=0)
    return {
        "dummy_stratified_macro_f1_mean": float(np.mean(stratified_scores)),
        "dummy_stratified_macro_f1_std": float(np.std(stratified_scores)),
        "dummy_prior_macro_f1": float(prior_metrics["macro_f1"]),
        "always_up_macro_f1": float(always_up_metrics["macro_f1"]),
        "always_down_macro_f1": float(always_down_metrics["macro_f1"]),
    }


def make_result_row(
    *,
    candidate_config: dict,
    model_name: str,
    ticker: str,
    seed: int,
    split: str,
    n_train_windows: int,
    n_val_windows: int,
    n_test_windows: int,
    label_retained_pct: float | None,
    model_metrics: dict,
    baseline_metrics: dict,
    confusion_matrix,
) -> dict:
    model_macro_f1 = model_metrics.get("macro_f1")
    dummy_macro_f1 = baseline_metrics.get("dummy_stratified_macro_f1_mean")
    delta_macro_f1 = None
    if model_macro_f1 is not None and dummy_macro_f1 is not None:
        delta_macro_f1 = float(model_macro_f1) - float(dummy_macro_f1)

    row = {
        "candidate_id": candidate_config["candidate_id"],
        "candidate_name": candidate_config["candidate_name"],
        "model_name": model_name,
        "ticker": ticker,
        "seed": seed,
        "window_size": int(candidate_config["window_size"]),
        "label_horizon_k": int(candidate_config["label_horizon_k"]),
        "threshold_bps": float(candidate_config["threshold_bps"]),
        "split": split,
        "n_train_windows": n_train_windows,
        "n_val_windows": n_val_windows,
        "n_test_windows": n_test_windows,
        "label_retained_pct": label_retained_pct,
        "model_macro_f1": model_macro_f1,
        "model_balanced_accuracy": model_metrics.get("balanced_accuracy"),
        "dummy_stratified_macro_f1_mean": dummy_macro_f1,
        "dummy_stratified_macro_f1_std": baseline_metrics.get("dummy_stratified_macro_f1_std"),
        "dummy_prior_macro_f1": baseline_metrics.get("dummy_prior_macro_f1"),
        "always_up_macro_f1": baseline_metrics.get("always_up_macro_f1"),
        "always_down_macro_f1": baseline_metrics.get("always_down_macro_f1"),
        "delta_macro_f1_vs_dummy": delta_macro_f1,
        "confusion_matrix": confusion_matrix,
    }
    missing_columns = [column for column in NOTEBOOK03_RESULT_COLUMNS if column not in row]
    if missing_columns:
        raise ValueError(f"Result row missing required columns: {missing_columns}")
    return {column: row[column] for column in NOTEBOOK03_RESULT_COLUMNS}

In [ ]:
NOTEBOOK03_OPTIONAL_DIAGNOSTIC_COLUMNS = [
    "candidate_id",
    "candidate_name",
    "train_up_pct",
    "train_down_pct",
    "val_up_pct",
    "val_down_pct",
    "test_up_pct",
    "test_down_pct",
    "label_n_total",
    "label_n_retained",
    "label_n_up",
    "label_n_down",
    "label_n_neutral",
    "label_n_cross_day",
    "label_n_tail",
    "model_precision_macro",
    "model_recall_macro",
    "best_epoch",
    "best_val_macro_f1",
    "training_time_seconds",
    "suspicious_status",
]


In [ ]:
optional_diagnostic_schema_df = pd.DataFrame(columns=NOTEBOOK03_OPTIONAL_DIAGNOSTIC_COLUMNS)
optional_diagnostic_schema_df.columns


In [ ]:
def _run_git(args: list[str]) -> str:
    completed = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed with code {completed.returncode}: {completed.stderr.strip()}"
        )
    return completed.stdout.strip()


def git_commit_hash() -> str:
    return _run_git(["rev-parse", "--short", "HEAD"])


def require_clean_git_status() -> None:
    status = _run_git(["status", "--short", "--untracked-files=all"])
    if status:
        raise RuntimeError(f"git status is not clean; refusing to run Notebook 03:\n{status}")


def require_full_run_enabled() -> None:
    if not FULL_RUN:
        raise RuntimeError(
            "FULL_RUN is False. Notebook 03 default mode is review-only and must not train."
        )


def require_training_enabled() -> None:
    if not RUN_TRAINING:
        raise RuntimeError(
            "RUN_TRAINING is False. Enable it only in an explicitly approved execution step."
        )


def selected_candidate_configs() -> list[dict]:
    by_id = {candidate["candidate_id"]: candidate for candidate in CANDIDATE_GRID}
    unknown = [candidate_id for candidate_id in SELECTED_CANDIDATES if candidate_id not in by_id]
    if unknown:
        raise ValueError(f"SELECTED_CANDIDATES contains unknown ids: {unknown}")
    return [copy.deepcopy(by_id[candidate_id]) for candidate_id in SELECTED_CANDIDATES]


def validate_scope_controls() -> dict:
    registry_names = {item["model_name"] for item in MODEL_REGISTRY}
    unknown_models = [model_name for model_name in SELECTED_MODELS if model_name not in registry_names]
    unknown_tickers = [ticker for ticker in SELECTED_TICKERS if ticker not in ALL_TICKERS]
    if unknown_models:
        raise ValueError(f"SELECTED_MODELS contains unknown names: {unknown_models}")
    if unknown_tickers:
        raise ValueError(f"SELECTED_TICKERS contains unknown tickers: {unknown_tickers}")
    if not SELECTED_SEEDS:
        raise ValueError("SELECTED_SEEDS must be non-empty")
    if MAX_RAW_ROWS_PER_TICKER is not None and MAX_RAW_ROWS_PER_TICKER <= 0:
        raise ValueError(
            f"MAX_RAW_ROWS_PER_TICKER must be None or > 0, got {MAX_RAW_ROWS_PER_TICKER}"
        )
    if MAX_EPOCHS <= 0:
        raise ValueError(f"MAX_EPOCHS must be > 0, got {MAX_EPOCHS}")
    if not selected_candidate_configs():
        raise ValueError("SELECTED_CANDIDATES must be non-empty")
    return {
        "selected_candidates": selected_candidate_configs(),
        "selected_models": list(SELECTED_MODELS),
        "selected_tickers": list(SELECTED_TICKERS),
        "selected_seeds": list(SELECTED_SEEDS),
        "max_raw_rows_per_ticker": MAX_RAW_ROWS_PER_TICKER,
        "max_epochs": MAX_EPOCHS,
    }


def require_data_available(tickers: list[str]) -> None:
    if not RAW_DATA_DIR.exists():
        raise FileNotFoundError(f"Missing raw data directory: {RAW_DATA_DIR}")
    missing_paths = [raw_path_for_ticker(ticker) for ticker in tickers if not raw_path_for_ticker(ticker).exists()]
    if missing_paths:
        raise FileNotFoundError(f"Missing raw data files: {missing_paths}")


def require_non_empty_windows(data_bundle: dict, candidate_config: dict) -> None:
    empty = []
    for split_name in ["train", "val", "test"]:
        for ticker, dataset in data_bundle["datasets"][split_name].items():
            if len(dataset) == 0:
                empty.append((candidate_config["candidate_id"], split_name, ticker))
    if empty:
        raise RuntimeError(f"Empty train/val/test windows found: {empty}")


def require_two_class_splits(data_bundle: dict, candidate_config: dict) -> None:
    one_class_splits = []
    for split_name in ["train", "val", "test"]:
        for ticker, labels in data_bundle["labels"][split_name].items():
            unique_labels = sorted(np.unique(labels).astype(int).tolist())
            if len(unique_labels) < 2:
                one_class_splits.append(
                    (candidate_config["candidate_id"], split_name, ticker, unique_labels)
                )
    if one_class_splits:
        raise RuntimeError(f"One-class train/val/test split labels found: {one_class_splits}")


def require_finite_metrics(metrics: dict, context: str) -> None:
    def check_value(name: str, value) -> None:
        if isinstance(value, dict):
            for child_name, child_value in value.items():
                check_value(f"{name}.{child_name}", child_value)
            return
        if isinstance(value, np.ndarray):
            if not np.isfinite(value.astype(float)).all():
                raise RuntimeError(f"{context} metric {name} contains NaN or inf")
            return
        if isinstance(value, (int, float, np.integer, np.floating)):
            if not np.isfinite(float(value)):
                raise RuntimeError(f"{context} metric {name} is NaN or inf: {value}")

    for metric_name, metric_value in metrics.items():
        check_value(metric_name, metric_value)


def require_not_suspicious_macro_f1(metrics: dict, context: str) -> None:
    macro_f1 = float(metrics["macro_f1"])
    if macro_f1 > SUSPICIOUS_MACRO_F1_THRESHOLD:
        raise RuntimeError(
            f"{context} macro_f1={macro_f1:.4f} exceeds {SUSPICIOUS_MACRO_F1_THRESHOLD:.2f}; "
            "stop and review for leakage risk"
        )


def require_expected_output_shape(model, loader, model_name: str) -> None:
    x, _ = next(iter(loader))
    model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x.to(DEVICE))
    expected_shape = (int(x.shape[0]), NUM_CLASSES)
    actual_shape = tuple(logits.shape)
    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{model_name} returned logits shape {actual_shape}; expected {expected_shape}"
        )


def label_retained_pct(label_diagnostics: dict) -> float:
    retained = label_diagnostics["n_up"] + label_diagnostics["n_down"]
    return float(retained / label_diagnostics["n_total"]) if label_diagnostics["n_total"] else np.nan


def run_id_or_default(run_id: str | None) -> str:
    if run_id:
        return run_id
    timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    return f"notebook03_{timestamp}_{git_commit_hash()}"


def planned_artifact_paths(run_id: str) -> dict[str, str]:
    run_dir = OUTPUT_DIR / run_id
    paths = {"run_dir": str(run_dir)}
    paths.update({key: str(run_dir / filename) for key, filename in ARTIFACT_NAMES.items()})
    return paths


def prepare_artifact_dir(run_id: str) -> Path:
    run_dir = OUTPUT_DIR / run_id
    if run_dir.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(
            f"Artifact directory already exists and ALLOW_OVERWRITE is False: {run_dir}"
        )
    run_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)
    return run_dir


def build_run_manifest(run_id: str, scope: dict, artifact_paths: dict[str, str]) -> dict:
    manifest = copy.deepcopy(RUN_MANIFEST_TEMPLATE)
    manifest["run_id"] = run_id
    manifest["timestamp"] = datetime.now(UTC).isoformat()
    manifest["git_commit_hash"] = git_commit_hash()
    manifest["guards"] = {
        "full_run": FULL_RUN,
        "run_training": RUN_TRAINING,
        "write_artifacts": WRITE_ARTIFACTS,
        "allow_overwrite": ALLOW_OVERWRITE,
    }
    manifest["scope"] = scope
    manifest["artifact_paths"] = artifact_paths
    return manifest


def write_run_artifacts(
    result_df: pd.DataFrame,
    diagnostic_df: pd.DataFrame,
    manifest: dict,
    artifact_paths: dict[str, str],
) -> None:
    if not WRITE_ARTIFACTS:
        return
    run_dir = Path(artifact_paths["run_dir"])
    if not run_dir.exists():
        raise RuntimeError(f"Artifact directory was not prepared: {run_dir}")
    result_df.to_csv(run_dir / ARTIFACT_NAMES["per_ticker_results"], index=False)
    result_df.groupby(["candidate_id", "model_name"], as_index=False).agg(
        model_macro_f1_mean=("model_macro_f1", "mean"),
        delta_macro_f1_vs_dummy_mean=("delta_macro_f1_vs_dummy", "mean"),
    ).to_csv(run_dir / ARTIFACT_NAMES["summary_by_model"], index=False)
    result_df.groupby(["candidate_id", "model_name", "seed"], as_index=False).agg(
        model_macro_f1_mean=("model_macro_f1", "mean"),
        delta_macro_f1_vs_dummy_mean=("delta_macro_f1_vs_dummy", "mean"),
    ).to_csv(run_dir / ARTIFACT_NAMES["summary_by_seed"], index=False)
    with open(run_dir / ARTIFACT_NAMES["run_manifest"], "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)


def run_one_model_ticker_seed(
    *,
    candidate_config: dict,
    model_name: str,
    ticker: str,
    seed: int,
    data_bundle: dict,
) -> tuple[dict, dict]:
    seed_everything(seed)
    train_dataset = data_bundle["datasets"]["train"][ticker]
    val_dataset = data_bundle["datasets"]["val"][ticker]
    test_dataset = data_bundle["datasets"]["test"][ticker]
    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_dataset, BATCH_SIZE, shuffle=False)
    test_loader = make_loader(test_dataset, BATCH_SIZE, shuffle=False)

    model = build_model(model_name, candidate_config)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    require_expected_output_shape(model, train_loader, model_name)

    best_epoch = None
    best_val_macro_f1 = None
    start_time = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=DEVICE,
        )
        require_finite_metrics(train_metrics, f"{candidate_config['candidate_id']} {model_name} {ticker} seed={seed} train epoch={epoch}")
        val_metrics, _, _ = evaluate(model=model, loader=val_loader, criterion=criterion, device=DEVICE)
        require_finite_metrics(val_metrics, f"{candidate_config['candidate_id']} {model_name} {ticker} seed={seed} val epoch={epoch}")
        require_not_suspicious_macro_f1(val_metrics, f"{candidate_config['candidate_id']} {model_name} {ticker} val")
        if best_val_macro_f1 is None or val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = float(val_metrics["macro_f1"])
            best_epoch = epoch

    test_metrics, y_test, y_pred = evaluate(model=model, loader=test_loader, criterion=criterion, device=DEVICE)
    require_finite_metrics(test_metrics, f"{candidate_config['candidate_id']} {model_name} {ticker} seed={seed} test")
    require_not_suspicious_macro_f1(test_metrics, f"{candidate_config['candidate_id']} {model_name} {ticker} test")

    y_train = data_bundle["labels"]["train"][ticker]
    baseline_metrics = summarize_dummy_baselines(y_train, y_test, seed)
    require_finite_metrics(baseline_metrics, f"{candidate_config['candidate_id']} {ticker} baselines")

    diagnostics = data_bundle["label_diagnostics"][ticker]
    result_row = make_result_row(
        candidate_config=candidate_config,
        model_name=model_name,
        ticker=ticker,
        seed=seed,
        split="test",
        n_train_windows=len(train_dataset),
        n_val_windows=len(val_dataset),
        n_test_windows=len(test_dataset),
        label_retained_pct=label_retained_pct(diagnostics),
        model_metrics=test_metrics,
        baseline_metrics=baseline_metrics,
        confusion_matrix=test_metrics["confusion_matrix"].tolist(),
    )
    diagnostic_row = {
        "candidate_id": candidate_config["candidate_id"],
        "candidate_name": candidate_config["candidate_name"],
        "train_up_pct": class_distribution(data_bundle["labels"]["train"][ticker])["up_pct"],
        "train_down_pct": class_distribution(data_bundle["labels"]["train"][ticker])["down_pct"],
        "val_up_pct": class_distribution(data_bundle["labels"]["val"][ticker])["up_pct"],
        "val_down_pct": class_distribution(data_bundle["labels"]["val"][ticker])["down_pct"],
        "test_up_pct": class_distribution(data_bundle["labels"]["test"][ticker])["up_pct"],
        "test_down_pct": class_distribution(data_bundle["labels"]["test"][ticker])["down_pct"],
        "label_n_total": diagnostics["n_total"],
        "label_n_retained": diagnostics["n_up"] + diagnostics["n_down"],
        "label_n_up": diagnostics["n_up"],
        "label_n_down": diagnostics["n_down"],
        "label_n_neutral": diagnostics["n_neutral"],
        "label_n_cross_day": diagnostics["n_cross_day"],
        "label_n_tail": diagnostics["n_tail"],
        "model_precision_macro": test_metrics["precision_macro"],
        "model_recall_macro": test_metrics["recall_macro"],
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "training_time_seconds": float(time.perf_counter() - start_time),
        "suspicious_status": False,
    }
    return result_row, diagnostic_row


def run_model_comparison(run_id: str | None = None) -> dict[str, pd.DataFrame | dict]:
    require_full_run_enabled()
    require_training_enabled()
    require_clean_git_status()
    scope = validate_scope_controls()
    require_data_available(scope["selected_tickers"])

    resolved_run_id = run_id_or_default(run_id)
    artifact_paths = planned_artifact_paths(resolved_run_id)
    if WRITE_ARTIFACTS:
        prepare_artifact_dir(resolved_run_id)

    result_rows = []
    diagnostic_rows = []
    for candidate_config in scope["selected_candidates"]:
        data_bundle = prepare_model_data(
            scope["selected_tickers"],
            candidate_config,
            max_raw_rows_per_ticker=scope["max_raw_rows_per_ticker"],
        )
        require_non_empty_windows(data_bundle, candidate_config)
        require_two_class_splits(data_bundle, candidate_config)
        for model_name in scope["selected_models"]:
            for ticker in scope["selected_tickers"]:
                for seed in scope["selected_seeds"]:
                    result_row, diagnostic_row = run_one_model_ticker_seed(
                        candidate_config=candidate_config,
                        model_name=model_name,
                        ticker=ticker,
                        seed=seed,
                        data_bundle=data_bundle,
                    )
                    result_rows.append(result_row)
                    diagnostic_rows.append(diagnostic_row)

    result_df = pd.DataFrame(result_rows, columns=NOTEBOOK03_RESULT_COLUMNS)
    diagnostic_df = pd.DataFrame(diagnostic_rows, columns=NOTEBOOK03_OPTIONAL_DIAGNOSTIC_COLUMNS)
    manifest = build_run_manifest(resolved_run_id, scope, artifact_paths)
    write_run_artifacts(result_df, diagnostic_df, manifest, artifact_paths)
    return {
        "results": result_df,
        "diagnostics": diagnostic_df,
        "manifest": manifest,
    }

In [ ]:
smoke_candidate = selected_candidate_configs()[0]
smoke_x = torch.randn(2, int(smoke_candidate["window_size"]), INPUT_SIZE)
smoke_rows = []
for model_name in SELECTED_MODELS:
    model = build_model(model_name, smoke_candidate)
    model.eval()
    with torch.no_grad():
        logits = model(smoke_x)
    output_shape = tuple(logits.shape)
    assert output_shape == (2, NUM_CLASSES), (
        f"{model_name} returned {output_shape}, expected {(2, NUM_CLASSES)}"
    )
    smoke_rows.append(
        {
            "candidate_id": smoke_candidate["candidate_id"],
            "model_name": model_name,
            "output_shape": output_shape,
        }
    )

smoke_shapes_df = pd.DataFrame(smoke_rows)
smoke_shapes_df

## Final Note

This notebook is patched to make the full entrypoint reviewable while keeping default execution closed. The next step should be a narrow, explicitly approved smoke that flips guards only for the requested scope.